# 🚀 Sổ Tay Cào Dữ Liệu Tốc Độ Cao & Ổn Định Trên Kaggle (Rock-Solid Aiohttp Crawler)

### 🌟 Kiến trúc tối ưu hóa độ ổn định & chống tràn RAM (Zero-OOM):
1. **Core Aiohttp Engine**: Thay thế `curl_cffi` bằng `aiohttp` thuần chuẩn AsyncIO, tiêu tốn **dưới 200 MB RAM**, triệt tiêu hoàn toàn lỗi **Exit code: 137 (OOM)**.
2. **Cơ Chế Phân Chia Batch (Chạy song song 2 Acc Kaggle)**:
   - Cấu hình `BATCH_SIZE = 10000`, `START_BATCH`, `MAX_BATCHES`.
   - **Acc 1**: `START_BATCH = 0`, `MAX_BATCHES = 50` (URLs 0 -> 500,000).
   - **Acc 2**: `START_BATCH = 50`, `MAX_BATCHES = 60` (URLs 500,000 -> 1,080,000).
3. **Domain Circuit Breaker Thông Minh**:
   - Tự động phát hiện domain chết / block / 403 / 429 sau **5 lỗi liên tiếp**.
   - Bỏ qua (Skip) ngay lập tức các URL còn lại của domain đó mà không tốn thời gian chờ mạng, đánh dấu mã lỗi rõ ràng.
4. **Chống Block & Hỗ Trợ Đa Dạng IP/Proxy**:
   - Xoay tua ngẫu nhiên danh sách User-Agent hiện đại.
   - Hỗ trợ gắn danh sách Proxy Pool để các request đi qua các IP khác nhau.
5. **Tự Động Resume & Xuất File Zip**:
   - Mỗi batch lưu riêng 1 file `.parquet`. Bị ngắt kết nối hay hết giờ cũng không mất dữ liệu.
   - Tự động nén thành file `.zip` để tải trực tiếp từ tab Output.

In [ ]:
# 1. Cài đặt các thư viện lõi tốc độ cao và ổn định
!pip install -q aiohttp aiohttp-socks trafilatura pyarrow nest_asyncio

In [ ]:
# 2. CẤU HÌNH BATCH & THÔNG SỐ (CHỈNH SỬA TẠI ĐÂY)
import os, glob

# ======================================================================
# ⚙️ CẤU HÌNH PHÂN CHIA BATCH (CHẠY SONG SONG 2 TÀI KHOẢN)
# ======================================================================
BATCH_SIZE = 10000      # 10,000 URLs / batch (Mức an toàn tuyệt đối cho RAM Kaggle)
START_BATCH = 0         # Acc 1: đặt là 0; Acc 2: đặt là 50 (hoặc batch tiếp theo)
MAX_BATCHES = 50        # Số batch cào trong phiên (50 x 10k = 500,000 URLs)
SPECIFIC_BATCHES = None # Ví dụ: [0, 1, 5] (nếu điền thì bỏ qua START_BATCH)

# ======================================================================
# 🛡️ CẤU HÌNH CONCURRENCY & CIRCUIT BREAKER
# ======================================================================
CONCURRENCY = 60        # Số request async song song (50 - 80)
MAX_DOMAIN_FAILS = 5    # Domain lỗi 5 lần liên tiếp -> Bật skip toàn bộ domain
TIMEOUT_SECONDS = 12    # Timeout tối đa cho 1 request mạng

# ======================================================================
# 🌐 CẤU HÌNH PROXY & ĐA DẠNG IP (NẾU CÓ)
# ======================================================================
PROXIES = []            # Điền danh sách proxy (dạng: 'http://ip:port' hoặc 'http://user:pass@ip:port')

# ======================================================================
# 📂 TỰ ĐỘNG TÌM ĐƯỜNG DẪN INPUT & OUTPUT
# ======================================================================
possible_inputs = glob.glob("/kaggle/input/**/remaining_urls.parquet", recursive=True)
if possible_inputs:
    INPUT_PATH = possible_inputs[0]
else:
    INPUT_PATH = "remaining_urls.parquet"

OUTPUT_DIR = "/kaggle/working/corpus_text"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📌 File Input: {INPUT_PATH}")
print(f"📌 Batch Size: {BATCH_SIZE:,} | Start: {START_BATCH} | Max Batches: {MAX_BATCHES}")
print(f"📌 Thư mục lưu kết quả: {OUTPUT_DIR}")

In [ ]:
# 3. ĐỊNH NGHĨA BỘ LỌC, CIRCUIT BREAKER & LOGIC CRAWLER
import sys, time, random, logging, asyncio, unicodedata, gc
import collections
from urllib.parse import urlparse
import pandas as pd
import aiohttp
import trafilatura
import nest_asyncio
nest_asyncio.apply()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)
# Tắt cảnh báo rác từ trafilatura
logging.getLogger('trafilatura').setLevel(logging.ERROR)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36 Edg/119.0.0.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
]

URL_BLACKLIST = [
    "/rao-vat/", "/san-pham/", "/gio-hang/", "/mua-ban/", "/cart/",
    "/checkout/", "/shop/", "/store/", "shopee", "lazada", "tiki"
]

MEDICAL_KEYWORDS = [
    "bác sĩ", "bệnh", "triệu chứng", "điều trị", "thuốc", "sức khỏe", "ung thư", "viêm", "đau", "chẩn đoán", "huyết áp", "phẫu thuật",
    "khám", "y tế", "bệnh nhân", "phòng khám", "dược", "tim mạch", "tiểu đường", "dinh dưỡng", "xét nghiệm", "nhi khoa", "sản khoa",
    "医院", "医生", "治疗", "疾病", "症状", "药", "患者", "健康", "诊断", "手术", "科",
    "doctor", "disease", "symptom", "treatment", "medicine", "health", "cancer", "pain", "diagnosis", "blood pressure", "surgery", "clinic",
    "medical", "patient", "hospital", "pharmacy", "syndrome", "infection", "therapy"
]

def check_layer_1(url):
    u = url.lower()
    for b in URL_BLACKLIST:
        if b in u: return f"Blacklist: {b}"
    return None

def check_layer_2(text):
    t = unicodedata.normalize("NFC", text.lower())
    for kw in MEDICAL_KEYWORDS:
        if unicodedata.normalize("NFC", kw) in t:
            return None
    return "No medical keyword"

# Trạng thái Circuit Breaker toàn cục cho các domain
domain_fail_counts = collections.defaultdict(int)
domain_circuit_broken = {}

def get_domain(url):
    try:
        return urlparse(url).netloc.lower()
    except Exception:
        return "unknown"

async def fetch_and_extract(session, url, row_id, sem):
    domain = get_domain(url)
    res = {'id': row_id, 'url': url, 'text': None, 'status': None, 'reason': '', 'dropped_at_layer': None}
    
    # 1. KIỂM TRA CIRCUIT BREAKER (SKIP NGAY NẾU DOMAIN ĐÃ BỊ ĐÁNH DẤU CHẾT)
    if domain in domain_circuit_broken:
        res['status'] = 'skipped'
        res['reason'] = f"Domain_Circuit_Broken: {domain_circuit_broken[domain]}"
        return res
        
    # 2. TẦNG 1: LỌC BLACKLIST URL
    l1 = check_layer_1(url)
    if l1:
        res['status'] = 'dropped'
        res['dropped_at_layer'] = 1
        res['reason'] = l1
        return res
        
    headers = {'User-Agent': random.choice(USER_AGENTS)}
    proxy = random.choice(PROXIES) if PROXIES else None
    
    async with sem:
        try:
            async with session.get(url, headers=headers, proxy=proxy, timeout=aiohttp.ClientTimeout(total=TIMEOUT_SECONDS)) as response:
                res['status'] = str(response.status)
                if response.status == 200:
                    domain_fail_counts[domain] = 0  # Reset chuỗi lỗi khi thành công
                    html = await response.text(errors='ignore')
                    
                    if "cf-browser-verification" in html or "Just a moment..." in html:
                        res['reason'] = "Cloudflare Challenge"
                        domain_fail_counts[domain] += 1
                    else:
                        text = trafilatura.extract(html, include_links=False, include_images=False)
                        if text and len(text.strip()) > 200:
                            l2 = check_layer_2(text)
                            if l2:
                                res['status'] = 'dropped'
                                res['dropped_at_layer'] = 2
                                res['reason'] = l2
                            else:
                                res['text'] = text
                                res['status'] = 'success'
                                res['reason'] = 'Success'
                        else:
                            res['reason'] = 'Empty text'
                else:
                    res['reason'] = f"HTTP_{response.status}"
                    if response.status in [403, 429, 500, 502, 503, 504]:
                        domain_fail_counts[domain] += 1
                        
        except asyncio.TimeoutError:
            res['status'] = 'error'
            res['reason'] = 'Timeout'
            domain_fail_counts[domain] += 1
        except Exception as e:
            err_name = type(e).__name__
            res['status'] = 'error'
            res['reason'] = f'Error: {err_name}'
            domain_fail_counts[domain] += 1
            
    # KIỂM TRA NGƯỠNG CIRCUIT BREAKER CHO DOMAIN (CHỈ LOG CẢNH BÁO 1 LẦN DUY NHẤT)
    if domain_fail_counts[domain] >= MAX_DOMAIN_FAILS and domain not in domain_circuit_broken:
        domain_circuit_broken[domain] = res['reason']
        logger.warning(f"🚨 [CIRCUIT BREAKER] Domain '{domain}' đã lỗi {MAX_DOMAIN_FAILS} lần liên tiếp -> Kích hoạt skip toàn bộ URLs còn lại của domain này ({res['reason']})")
        
    return res

async def process_batch(df_batch, batch_index, output_dir, concurrency):
    out_file = os.path.join(output_dir, f"corpus_part_{batch_index:05d}.parquet")
    if os.path.exists(out_file):
        logger.info(f"⏩ [SKIP] Batch {batch_index:05d} đã có sẵn file kết quả. Bỏ qua.")
        return
        
    sem = asyncio.Semaphore(concurrency)
    connector = aiohttp.TCPConnector(limit=concurrency, ssl=False)
    
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [
            fetch_and_extract(session, row['url'], row['id'], sem)
            for _, row in df_batch.iterrows()
        ]
        
        results = []
        total = len(tasks)
        start_t = time.time()
        
        for future in asyncio.as_completed(tasks):
            r = await future
            results.append(r)
            if len(results) % 500 == 0 or len(results) == total:
                speed = len(results) / (time.time() - start_t) if time.time() > start_t else 0
                logger.info(f"   Batch {batch_index:05d}: {len(results):,}/{total:,} ({len(results)/total*100:.1f}%) | Tốc độ: {speed:.1f} URL/s | Skipped Domains: {len(domain_circuit_broken)}")
                
    # Tách và lưu trữ kết quả an toàn
    df_res = pd.DataFrame(results)
    df_res.to_parquet(out_file, index=False)
    
    success_cnt = (df_res['status'] == 'success').sum()
    skipped_cnt = (df_res['status'] == 'skipped').sum()
    logger.info(f"💾 Đã lưu xong Batch {batch_index:05d} -> {out_file} (Success: {success_cnt:,}, Skipped: {skipped_cnt:,})")
    
    del results, df_res
    gc.collect()

In [ ]:
# 4. THỰC THI CHẠY TỪNG BATCH TUẦN TỰ (AN TOÀN BỘ NHỚ TUYỆT ĐỐI)
print(f"Đang đọc file gốc: {INPUT_PATH}")
df_full = pd.read_parquet(INPUT_PATH)
total_full = len(df_full)
total_batches_possible = (total_full // BATCH_SIZE) + (1 if total_full % BATCH_SIZE > 0 else 0)
print(f"Tổng số URLs: {total_full:,} | Kích thước batch: {BATCH_SIZE:,} | Tổng số batches: {total_batches_possible}")

if SPECIFIC_BATCHES is not None:
    batch_list = SPECIFIC_BATCHES
else:
    end_b = min(START_BATCH + MAX_BATCHES, total_batches_possible) if MAX_BATCHES else total_batches_possible
    batch_list = list(range(START_BATCH, end_b))

print(f"Danh sách batch cần xử lý phiên này ({len(batch_list)} batches): {batch_list[0]} -> {batch_list[-1]}")

for b_idx in batch_list:
    out_file = os.path.join(OUTPUT_DIR, f"corpus_part_{b_idx:05d}.parquet")
    if os.path.exists(out_file):
        print(f"⏩ [SKIP] Batch {b_idx:05d} đã hoàn tất trước đó: {out_file}")
        continue
        
    s_idx = b_idx * BATCH_SIZE
    e_idx = min(s_idx + BATCH_SIZE, total_full)
    df_batch = df_full.iloc[s_idx:e_idx].copy()
    print(f"\n🚀 [BẮT ĐẦU] Batch {b_idx:05d} (Dòng {s_idx:,} -> {e_idx:,}) | {len(df_batch):,} URLs")
    
    # Chạy cào batch bằng aiohttp mượt mà
    asyncio.run(process_batch(df_batch, b_idx, OUTPUT_DIR, CONCURRENCY))
    
    del df_batch
    gc.collect()

print("\n🎉 HOÀN TẤT TOÀN BỘ CÁC BATCH ĐƯỢC GIAO!")

In [ ]:
# 5. NÉN STREAM ZIP TẤT CẢ CÁC BATCH ĐÃ HOÀN TẤT (0 MB RAM)
import zipfile

completed_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "corpus_part_*.parquet")))
print(f"Tìm thấy {len(completed_files)} file batch đã hoàn tất trong {OUTPUT_DIR}")

if completed_files:
    zip_name = f"/kaggle/working/corpus_text_{START_BATCH}_to_{batch_list[-1]}.zip"
    print(f"Đang nén các file vào {zip_name}...")
    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for f in completed_files:
            zf.write(f, arcname=os.path.basename(f))
            
    size_mb = os.path.getsize(zip_name) / 1024 / 1024
    print(f"\n🎁 ĐÃ TẠO FILE ZIP THÀNH CÔNG: {zip_name} ({size_mb:.2f} MB)")
    print("👉 Bạn có thể tải trực tiếp file zip này từ tab Kaggle Output!")
else:
    print("Chưa có file batch nào hoàn tất để nén.")